# ARGCN 数据处理教程

## 概述

本 notebook 展示 ARGCN 教学版的**完整数据处理流程**。这里的目标不是“黑盒调用原仓库 parser”，而是把
`reaction-gcnn/dataset/suzuki_data_frame_parser.py` 背后的关键步骤拆开，用最小化的 Python + RDKit 代码重新说明：

1. 原始反应记录长什么样
2. 反应如何拆成 `Reactant1 / Reactant2 / Product`
3. 条件名称如何编码成多热标签向量
4. 分子如何转成关系图卷积需要的 `atom_ids + relation_adjs`
5. 多个样本如何拼成模型输入 batch

### 本 notebook 内容

| 章节 | 内容 |
|------|------|
| 1 | 读取微型教学数据集 |
| 2 | 对照 Suzuki 字典构造标签空间 |
| 3 | 解析反应 SMILES |
| 4 | 生成仓库风格的标签字段 |
| 5 | 把每个分子转成关系图张量 |
| 6 | 构造 ARGCN batch |


In [1]:
# ====== Step 0: 环境初始化 ======
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from rdkit import Chem

def find_project_root(start=None):
    here = Path(start or os.getcwd()).resolve()
    for path in [here, *here.parents]:
        if (path / 'teaching_demos').exists() and (path / 'source_repos').exists():
            return path
    raise RuntimeError('未找到项目根目录')

PROJECT_ROOT = find_project_root()
TUTORIAL_DIR = PROJECT_ROOT / 'teaching_demos' / '4.reaction_condition_tutorial' / 'ARGCN'
REPO_DIR = PROJECT_ROOT / 'source_repos' / 'reaction-gcnn'
DEMO_DATA = TUTORIAL_DIR / 'data' / 'demo_data.csv'
DICT_PATH = REPO_DIR / 'data' / 'all_dictionaries' / 'suzuki_dict.csv'
REPO_STYLE_CSV = TUTORIAL_DIR / 'data' / 'demo_repo_style.csv'

print('DEMO_DATA    =', DEMO_DATA)
print('DICT_PATH    =', DICT_PATH)
print('REPO_STYLE_CSV =', REPO_STYLE_CSV)


DEMO_DATA    = /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/4.reaction_condition_tutorial/ARGCN/data/demo_data.csv
DICT_PATH    = /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/reaction-gcnn/data/all_dictionaries/suzuki_dict.csv
REPO_STYLE_CSV = /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/4.reaction_condition_tutorial/ARGCN/data/demo_repo_style.csv


## 1. 读取微型教学数据集

### 为什么先看这个表？

原仓库最终读入的是已经清洗好的 CSV，但真正的教学起点应该更早：

- 一条反应记录包含 `reaction_smiles`
- 条件列还是**人类可读的名称**
- 多溶剂、多添加剂会以 `|` 分隔

这更接近你自己手头实验数据的初始形态。

### 源码对应

- 数据入口: `source_repos/reaction-gcnn/train.py`
- 解析器: `source_repos/reaction-gcnn/dataset/suzuki_csv_file_parser.py`
- 真正的逻辑主体: `source_repos/reaction-gcnn/dataset/suzuki_data_frame_parser.py`


In [2]:
# ====== Step 1: 读取微型教学数据 ======
demo_df = pd.read_csv(DEMO_DATA)
demo_df


,reaction_id,reaction_smiles,yield,metal,ligand,base,solvent,additive
0,rxn_001,Brc1ccccc1.OB(O)c1ccccc1>>c1ccc(-c2ccccc2)cc1,0.82,tetrakis(triphenylphosphine) palladium(0),triphenylphosphine,potassium carbonate,tetrahydrofuran,tetrabutylammomium bromide
1,rxn_002,Brc1ccc(OC)cc1.OB(O)c1ccncc1>>COc1ccc(-c2ccncc...,0.74,palladium diacetate,XPhos,potassium phosphate,ethanol|water,water
2,rxn_003,Brc1ccccn1.Cc1ccc(B(O)O)cc1>>Cc1ccc(-c2ccccn2)cc1,0.67,"(1,1'-bis(diphenylphosphino)ferrocene)palladiu...","1,1'-bis-(diphenylphosphino)ferrocene",caesium carbonate,"1,4-dioxane|water",NaN


## 2. 读取 Suzuki 条件字典并构造全局标签空间

`reaction-gcnn` 不是分别做“金属分类”“配体分类”“碱分类”……，而是把这些条件家族拼接成一个**统一的多热向量**。

对 Suzuki 数据，`train.py` 中给出的家族规模是：

- 金属 `M`: 28
- 配体 `L`: 23
- 碱 `B`: 35
- 溶剂 `S`: 10
- 添加剂 `A`: 17

总计 113 个真实条件槽位。原 parser 还会额外保留：

- 1 个保留槽位（教学上可以理解为“预留/unknown”）
- 每个条件家族 1 个“空条件”槽位，共 5 个

因此最终 `class_num = 119`。

### 源码对应

- `source_repos/reaction-gcnn/train.py` 中的 `class_dict`
- `source_repos/reaction-gcnn/dataset/suzuki_data_frame_parser.py` 中的 `rea_cat` 构造逻辑


In [3]:
# ====== Step 2: 构造全局标签映射 ======
suzuki_dict_df = pd.read_csv(DICT_PATH)

FAMILY_META = {
    'M': ('metal_name', 'metal_bin'),
    'L': ('ligand_name', 'ligand_bin'),
    'B': ('base_name', 'base_bin'),
    'S': ('solvent_name', 'solvent_bin'),
    'A': ('additive_name', 'additive_bin'),
}
RAW_COL_TO_FAMILY = {
    'metal': 'M',
    'ligand': 'L',
    'base': 'B',
    'solvent': 'S',
    'additive': 'A',
}

def extract_family_entries(df, family, name_col, bin_col):
    part = df[[name_col, bin_col]].dropna().copy()
    part[bin_col] = part[bin_col].astype(str)
    part = part[part[bin_col].str.startswith(family)]
    part['local_index'] = part[bin_col].str[1:].astype(int) - 1
    part = part.drop_duplicates(subset=[bin_col]).sort_values('local_index')
    return part

family_offsets = {}
family_name_to_global = {}
global_index_to_name = {}
total_real_slots = 0
rows = []

for family, (name_col, bin_col) in FAMILY_META.items():
    entries = extract_family_entries(suzuki_dict_df, family, name_col, bin_col)
    family_offsets[family] = total_real_slots
    family_name_to_global[family] = {}
    for _, row in entries.iterrows():
        global_index = total_real_slots + int(row['local_index'])
        name = str(row[name_col]).strip()
        family_name_to_global[family][name] = global_index
        global_index_to_name[global_index] = name
    rows.append({
        'family': family,
        'num_labels': len(entries),
        'global_start': total_real_slots,
        'global_end': total_real_slots + len(entries) - 1,
    })
    total_real_slots += len(entries)

reserved_unknown_index = total_real_slots
null_condition_index = {family: total_real_slots + 1 + i for i, family in enumerate(FAMILY_META)}
class_num = total_real_slots + 1 + len(FAMILY_META)

summary_df = pd.DataFrame(rows)
print('family summary:')
display(summary_df)
print('total_real_slots      =', total_real_slots)
print('reserved_unknown_index =', reserved_unknown_index)
print('null_condition_index   =', null_condition_index)
print('class_num              =', class_num)


family summary:


,family,num_labels,global_start,global_end
0,M,28,0,27
1,L,23,28,50
2,B,35,51,85
3,S,10,86,95
4,A,17,96,112


total_real_slots      = 113
reserved_unknown_index = 113
null_condition_index   = {'M': 114, 'L': 115, 'B': 116, 'S': 117, 'A': 118}
class_num              = 119


## 3. 解析反应 SMILES

训练脚本最终把三种分子作为三个独立输入：

- `Reactant1`
- `Reactant2`
- `Product`

这正对应 `train.py` 里的：

```python
smiles_col=['Reactant1', 'Reactant2', 'Product']
```

因此我们要先把一条 `reaction_smiles` 拆成这三列，然后做基础标准化。

### 源码对应

- `source_repos/reaction-gcnn/train.py`
- `source_repos/reaction-gcnn/dataset/suzuki_data_frame_parser.py`


In [4]:
# ====== Step 3: 解析 reaction_smiles 并标准化 ======
def canonicalize_smiles(smiles):
    if pd.isna(smiles) or not str(smiles).strip():
        return None
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        raise ValueError(f'无法解析 SMILES: {smiles}')
    return Chem.MolToSmiles(mol, canonical=True)

def split_reaction_smiles(reaction_smiles):
    reactants, product = reaction_smiles.split('>>')
    reactant_parts = [part for part in reactants.split('.') if part]
    if len(reactant_parts) == 1:
        reactant_parts.append(None)
    if len(reactant_parts) != 2:
        raise ValueError(f'教学版当前假设每条反应有两个底物，实际得到: {reactant_parts}')
    return {
        'Reactant1': canonicalize_smiles(reactant_parts[0]),
        'Reactant2': canonicalize_smiles(reactant_parts[1]) if reactant_parts[1] else None,
        'Product': canonicalize_smiles(product),
    }

split_df = demo_df['reaction_smiles'].apply(split_reaction_smiles).apply(pd.Series)
parsed_df = pd.concat([demo_df, split_df], axis=1)
parsed_df[['reaction_id', 'Reactant1', 'Reactant2', 'Product', 'yield']]


,reaction_id,Reactant1,Reactant2,Product,yield
0,rxn_001,Brc1ccccc1,OB(O)c1ccccc1,c1ccc(-c2ccccc2)cc1,0.82
1,rxn_002,COc1ccc(Br)cc1,OB(O)c1ccncc1,COc1ccc(-c2ccncc2)cc1,0.74
2,rxn_003,Brc1ccccn1,Cc1ccc(B(O)O)cc1,Cc1ccc(-c2ccccn2)cc1,0.67


## 4. 条件名称编码为仓库风格的标签字段

原 parser 期待的标签列不是名称，而是类似：

- `M = "[0]"`
- `L = "[28]"`
- `S = "[86, 94]"`

也就是说：

1. 名称先根据字典转成全局索引
2. 每个条件家族允许多个索引
3. 最终写成字符串形式的 Python list

下面这一步会同时导出一个教学版的 `demo_repo_style.csv`，方便你对照原仓库输入格式。


In [5]:
# ====== Step 4: 条件名称 -> 全局索引 -> 仓库风格字段 ======
def parse_condition_names(raw_value):
    if pd.isna(raw_value) or not str(raw_value).strip():
        return []
    return [item.strip() for item in str(raw_value).split('|') if item.strip()]

def encode_condition_names(row):
    encoded = {}
    for raw_col, family in RAW_COL_TO_FAMILY.items():
        names = parse_condition_names(row[raw_col])
        global_indices = []
        for name in names:
            if name not in family_name_to_global[family]:
                raise KeyError(f'{family} 家族中找不到条件名称: {name}')
            global_indices.append(family_name_to_global[family][name])
        encoded[family] = global_indices
    return encoded

def row_to_repo_style(row):
    encoded = encode_condition_names(row)
    result = {
        'id': row['reaction_id'],
        'Yield': row['yield'],
        'Reactant1': row['Reactant1'],
        'Reactant2': row['Reactant2'],
        'Product': row['Product'],
    }
    for family in FAMILY_META:
        result[family] = str(encoded[family])
    return result

repo_style_df = parsed_df.apply(row_to_repo_style, axis=1, result_type='expand')
repo_style_df.to_csv(REPO_STYLE_CSV, sep=';', index=False)

print('导出的仓库风格文件:', REPO_STYLE_CSV.relative_to(PROJECT_ROOT))
repo_style_df


导出的仓库风格文件: teaching_demos/4.reaction_condition_tutorial/ARGCN/data/demo_repo_style.csv


,id,Yield,Reactant1,Reactant2,Product,M,L,B,S,A
0,rxn_001,0.82,Brc1ccccc1,OB(O)c1ccccc1,c1ccc(-c2ccccc2)cc1,[0],[28],[51],[86],[96]
1,rxn_002,0.74,COc1ccc(Br)cc1,OB(O)c1ccncc1,COc1ccc(-c2ccncc2)cc1,[1],[30],[53],"[87, 94]",[97]
2,rxn_003,0.67,Brc1ccccn1,Cc1ccc(B(O)O)cc1,Cc1ccc(-c2ccccn2)cc1,[2],[36],[54],"[89, 94]",[]


## 5. 把分子转换为 RelGCN 需要的图结构

`models/relgcn.py` 的输入核心是两部分：

- `atom_ids`: 每个原子的原子序数
- `adj`: 形状为 `(num_edge_type, num_nodes, num_nodes)` 的关系邻接张量

对于这个仓库，`num_edge_type = 4`，分别表示：

1. 单键
2. 双键
3. 三键
4. 芳香键

### 源码对应

- `source_repos/reaction-gcnn/models/relgcn.py`
- `source_repos/reaction-gcnn/train_attention_model.py` 中 `num_edge_type = 4`


In [6]:
# ====== Step 5: SMILES -> RelGCN 图张量 ======
BOND_TYPE_TO_CHANNEL = {
    Chem.rdchem.BondType.SINGLE: 0,
    Chem.rdchem.BondType.DOUBLE: 1,
    Chem.rdchem.BondType.TRIPLE: 2,
    Chem.rdchem.BondType.AROMATIC: 3,
}

def smiles_to_relgcn_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'无法解析分子: {smiles}')

    atom_ids = np.array([atom.GetAtomicNum() for atom in mol.GetAtoms()], dtype=np.int64)
    num_atoms = len(atom_ids)
    adj = np.zeros((4, num_atoms, num_atoms), dtype=np.float32)

    for bond in mol.GetBonds():
        begin = bond.GetBeginAtomIdx()
        end = bond.GetEndAtomIdx()
        channel = BOND_TYPE_TO_CHANNEL.get(bond.GetBondType())
        if channel is None:
            continue
        adj[channel, begin, end] = 1.0
        adj[channel, end, begin] = 1.0

    mask = np.ones(num_atoms, dtype=np.float32)
    return {
        'atom_ids': atom_ids,
        'adj': adj,
        'mask': mask,
        'num_atoms': num_atoms,
    }

graph_example = smiles_to_relgcn_graph(repo_style_df.loc[0, 'Reactant1'])
print('example smiles:', repo_style_df.loc[0, 'Reactant1'])
print('atom_ids shape:', graph_example['atom_ids'].shape)
print('adj shape     :', graph_example['adj'].shape)
print('num_atoms     :', graph_example['num_atoms'])
print('atom_ids      :', graph_example['atom_ids'])


example smiles: Brc1ccccc1
atom_ids shape: (7,)
adj shape     : (4, 7, 7)
num_atoms     : 7
atom_ids      : [35  6  6  6  6  6  6]


## 6. 组装成 ARGCN 教学版样本与 batch

到这里，我们已经有了：

- 三个分子的图结构
- 一个样本的多热标签向量

剩下的事情就是把多个样本 pad 成 batch。这一步相当于教学版对原仓库 `concat_mols` 的手工复现。

### 输出张量说明

| 键 | 含义 |
|----|------|
| `atom_ids_1 / atom_ids_2 / atom_ids_3` | 三个分子的原子编号张量 |
| `adjs_1 / adjs_2 / adjs_3` | 三个分子的关系邻接张量 |
| `mask_1 / mask_2 / mask_3` | pad 位置掩码 |
| `labels` | 长度为 119 的多热向量 |
| `yields` | 反应收率 |


In [7]:
# ====== Step 6: 构造样本对象并拼 batch ======
def encode_multihot_target(encoded_conditions):
    target = np.zeros(class_num, dtype=np.float32)
    for family in FAMILY_META:
        indices = encoded_conditions[family]
        if indices:
            target[indices] = 1.0
        else:
            target[null_condition_index[family]] = 1.0
    return target

def build_argcn_item(row):
    encoded = encode_condition_names(row)
    return {
        'reaction_id': row['reaction_id'],
        'yield_value': float(row['yield']),
        'graphs': [
            smiles_to_relgcn_graph(row['Reactant1']),
            smiles_to_relgcn_graph(row['Reactant2']),
            smiles_to_relgcn_graph(row['Product']),
        ],
        'encoded_conditions': encoded,
        'label_vector': encode_multihot_target(encoded),
    }

def pad_atom_ids(atom_ids, max_atoms):
    padded = np.zeros(max_atoms, dtype=np.int64)
    padded[:len(atom_ids)] = atom_ids
    return padded

def pad_mask(mask, max_atoms):
    padded = np.zeros(max_atoms, dtype=np.float32)
    padded[:len(mask)] = mask
    return padded

def pad_adj(adj, max_atoms):
    padded = np.zeros((adj.shape[0], max_atoms, max_atoms), dtype=np.float32)
    n = adj.shape[1]
    padded[:, :n, :n] = adj
    return padded

def collate_argcn_batch(items):
    batch = {'reaction_ids': [item['reaction_id'] for item in items]}
    for mol_idx in range(3):
        max_atoms = max(item['graphs'][mol_idx]['num_atoms'] for item in items)
        batch[f'atom_ids_{mol_idx+1}'] = torch.tensor(
            np.stack([pad_atom_ids(item['graphs'][mol_idx]['atom_ids'], max_atoms) for item in items]),
            dtype=torch.long,
        )
        batch[f'adjs_{mol_idx+1}'] = torch.tensor(
            np.stack([pad_adj(item['graphs'][mol_idx]['adj'], max_atoms) for item in items]),
            dtype=torch.float32,
        )
        batch[f'mask_{mol_idx+1}'] = torch.tensor(
            np.stack([pad_mask(item['graphs'][mol_idx]['mask'], max_atoms) for item in items]),
            dtype=torch.float32,
        )
    batch['labels'] = torch.tensor(np.stack([item['label_vector'] for item in items]), dtype=torch.float32)
    batch['yields'] = torch.tensor([item['yield_value'] for item in items], dtype=torch.float32)
    return batch

items = [build_argcn_item(row) for _, row in parsed_df.iterrows()]
batch = collate_argcn_batch(items)

print('reaction_ids:', batch['reaction_ids'])
for key, value in batch.items():
    if isinstance(value, torch.Tensor):
        print(f'{key:12s} -> shape={tuple(value.shape)}, dtype={value.dtype}')


reaction_ids: ['rxn_001', 'rxn_002', 'rxn_003']
atom_ids_1   -> shape=(3, 9), dtype=torch.int64
adjs_1       -> shape=(3, 4, 9, 9), dtype=torch.float32
mask_1       -> shape=(3, 9), dtype=torch.float32
atom_ids_2   -> shape=(3, 10), dtype=torch.int64
adjs_2       -> shape=(3, 4, 10, 10), dtype=torch.float32
mask_2       -> shape=(3, 10), dtype=torch.float32
atom_ids_3   -> shape=(3, 14), dtype=torch.int64
adjs_3       -> shape=(3, 4, 14, 14), dtype=torch.float32
mask_3       -> shape=(3, 14), dtype=torch.float32
labels       -> shape=(3, 119), dtype=torch.float32
yields       -> shape=(3,), dtype=torch.float32


## 完整流程总结

### 数据处理管线回顾

```text
demo_data.csv
    ↓
解析 reaction_smiles
    ↓
Reactant1 / Reactant2 / Product
    ↓
条件名称映射到全局索引
    ↓
构造 119 维多热标签
    ↓
每个分子转成 (atom_ids, relation_adjs, mask)
    ↓
pad 成 batch
```

### 教学版与原仓库的差异

- 原仓库通过 `chainer-chemistry` 的 preprocessor 自动完成部分图特征提取。
- 教学版显式地把每一步写出来，便于观察张量结构。
- 原仓库里还包含缓存、更多数据集分支、训练/验证拆分等工程细节，这里暂时省略。

下一步建议打开 `3_模型展示.ipynb`，把这些张量如何进入 Torch 版 ARGCN 模型看清楚。
